# Satellite-Derived Chlorophyll Time Series for Lakes: MODIS 500 m

## Overview

This notebook extracts chlorophyll-a concentration estimates and Normalized Difference Chlorophyll Index (NDCI) values from MODIS (Terra and Aqua) specified lake locations.

MODIS (Moderate Resolution Imaging Spectroradiometer) provides daily global coverage in the visible bands at 250, 500, and 1,000 m resolution from two satellites: Terra (morning overpass) and Aqua (afternoon overpass). This notebook derives chlorophyll-a concentrations using an empirical green-to-red ratio algorithm using the 500 m bands.

- Purpose: Generate time series of chlorophyll indices from satellite imagery
- Study Areas: Detroit Lake and Upper Klamath Lake
- Satellite Sensors: MODIS-Aqua and MODIS-Terra
- Resolution: 500 m
- Output: CSV files with date-stamped chlorophyll index values

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

In [2]:
# -----------------------------
# User parameters
# -----------------------------

lakes = [
    dict(
        name='Detroit',
        lon=-122.184, lat=44.711,
        terra_export='Detroit_MODIS_Terra_Chlorophyll_B4_Green_B1_Red_500m',
        aqua_export='Detroit_MODIS_Aqua_Chlorophyll_B4_Green_B1_Red_500m',
        # Set to None to skip Chl computation (export ratio/log_ratio only)
        a=None, b=None
    ),
    dict(
        name='UpperKlamath',
        lon=-121.900, lat=42.400,
        terra_export='Klamath_MODIS_Terra_Chlorophyll_B4_Green_B1_Red_500m',
        aqua_export='Klamath_MODIS_Aqua_Chlorophyll_B4_Green_B1_Red_500m',
        a=None, b=None
    ),
]

start_date = '2011-01-01'
end_date   = '2025-12-31'

# If a lake has a/b=None, Chl-a won't be computed (only ratio/log_ratio exported)
DEFAULT_A = None
DEFAULT_B = None

# ROI & masking
ROI_RADIUS_M         = 1000    # offshore sampling radius (m)
SHORELINE_BUFFER_M   = 500     # erode water mask away from land (m)
WATER_OCC_THRESHOLD  = 75      # JRC occurrence threshold (0-100)
GSW_DATASET_ID       = 'JRC/GSW1_4/GlobalSurfaceWater'  # use '...1_3...' if needed

# MODIS collections and bands
TERRA_COL_ID = 'MODIS/061/MOD09GA'
AQUA_COL_ID  = 'MODIS/061/MYD09GA'
B_GREEN      = 'sur_refl_b04'  # ~555 nm (500 m)
B_RED        = 'sur_refl_b01'  # ~645 nm (500 m)
SR_SCALE     = 1e-4            # scale factor

In [3]:
# -----------------------------
# QA and water masks
# -----------------------------

def mask_mod09(img):
    """
    MOD09/MYD09 QA-based mask using state_1km:
      - Cloud state (bits 0-1): mask cloudy or mixed
      - Cloud shadow (bit 2)
      - Cirrus (bits 8-9)
      - Internal cloud (bit 10)
      - Snow/ice (bit 12)
    """
    qa = img.select('state_1km')
    cloud_state = qa.bitwiseAnd(3)                     # bits 0-1
    cloudy_or_mixed = cloud_state.eq(1).Or(cloud_state.eq(2))
    shadow   = qa.bitwiseAnd(1 << 2).neq(0)            # bit 2
    cirrus   = qa.bitwiseAnd(3 << 8).neq(0)            # bits 8-9
    intcloud = qa.bitwiseAnd(1 << 10).neq(0)           # bit 10
    snowice  = qa.bitwiseAnd(1 << 12).neq(0)           # bit 12

    mask = cloudy_or_mixed.Or(shadow).Or(cirrus).Or(intcloud).Or(snowice).Not()
    return img.updateMask(mask)

def build_water_mask():
    """
    Persistent open-water mask from JRC Global Surface Water 'occurrence'.
    Erode by SHORELINE_BUFFER_M to mitigate shoreline adjacency.
    """
    gsw = ee.Image(GSW_DATASET_ID).select('occurrence')
    water = gsw.gte(WATER_OCC_THRESHOLD)
    water_eroded = water.focal_min(radius=SHORELINE_BUFFER_M, units='meters')
    return water_eroded

WATER_MASK = build_water_mask()

# -----------------------------------------------------------
# Statistics and optional chlorophyll computation, per image
# -----------------------------------------------------------

def per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff):
    """
    For a single image:
      - Apply QA & water mask
      - Compute ratio = SR555/SR645 and log_ratio = log10(ratio)
      - Optionally compute Chl-a if a_coeff and b_coeff are not None
      - Reduce over ROI (median) and return a Feature with properties
    """
    img = mask_mod09(img).updateMask(WATER_MASK)

    g = img.select(B_GREEN).multiply(SR_SCALE)
    r = img.select(B_RED).multiply(SR_SCALE)

    valid = g.gt(0).And(r.gt(0))
    ratio_img = g.divide(r).updateMask(valid).rename('ratio')
    log_ratio_img = ratio_img.log10().rename('log_ratio')

    # Reduce ratio and log_ratio over ROI
    ratio_stats = ratio_img.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=500,
        maxPixels=1e9,
        bestEffort=True
    )
    log_ratio_stats = log_ratio_img.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=roi_geom,
        scale=500,
        maxPixels=1e9,
        bestEffort=True
    )

    props = ee.Dictionary({
        'datetime': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss'),
        'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
        'time': ee.Date(img.get('system:time_start')).format('HH:mm:ss'),
        'sensor': sensor_tag,
        'ratio': ratio_stats.get('ratio'),
        'log_ratio': log_ratio_stats.get('log_ratio')
    })

    # Optionally compute Chl-a
    def add_chl_props(p):
        log10_chl = log_ratio_img.multiply(a_coeff).add(b_coeff)
        chl_img = ee.Image(10).pow(log10_chl).rename('chlor_a')
        chl_stats = chl_img.reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=roi_geom,
            scale=500,
            maxPixels=1e9,
            bestEffort=True
        )
        return ee.Dictionary(p).set('chlor_a', chl_stats.get('chlor_a'))

    if (a_coeff is not None) and (b_coeff is not None):
        props = add_chl_props(props)

    return ee.Feature(None, props)

def imagecollection_to_features(col_id, roi_geom, sensor_tag, a_coeff, b_coeff):
    ic = (ee.ImageCollection(col_id)
          .filterDate(start_date, end_date)
          .filterBounds(roi_geom))

    fc = ic.map(lambda img: per_image_stats(img, roi_geom, sensor_tag, a_coeff, b_coeff))
    # Require ratio to exist (i.e., non-null after masking)
    fc = fc.filter(ee.Filter.notNull(['ratio']))
    return fc

In [4]:
# --------------------------------------
# Main loop with client-side CSV export
# --------------------------------------

for lake in lakes:
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(ROI_RADIUS_M)

    a = lake.get('a', DEFAULT_A)
    b = lake.get('b', DEFAULT_B)

    terra_fc = imagecollection_to_features(TERRA_COL_ID, roi, 'Terra', a, b)
    aqua_fc  = imagecollection_to_features(AQUA_COL_ID,  roi, 'Aqua',  a, b)

    # Availability
    print(f"{lake['name']} Terra features =", terra_fc.size().getInfo())
    print(f"{lake['name']} Aqua  features =", aqua_fc.size().getInfo())

    # ---------- Client-side export ----------
    # Terra
    terra_rows = terra_fc.getInfo()['features']
    terra_records = [f['properties'] for f in terra_rows]
    terra_df = pd.DataFrame.from_records(terra_records)
    terra_df = terra_df.sort_values('datetime')
    terra_df.to_csv(lake['terra_export'] + '.csv', index=False)

    # Aqua
    aqua_rows = aqua_fc.getInfo()['features']
    aqua_records = [f['properties'] for f in aqua_rows]
    aqua_df = pd.DataFrame.from_records(aqua_records)
    aqua_df = aqua_df.sort_values('datetime')
    aqua_df.to_csv(lake['aqua_export'] + '.csv', index=False)

print("Done.")

Detroit Terra features = 1931
Detroit Aqua  features = 2002
UpperKlamath Terra features = 2499
UpperKlamath Aqua  features = 2404
Done.
